# Step 1: Synthetic Data Generator

This notebook creates synthetic user activity data that mimics the CERT 4.2 insider threat dataset.

It simulates:
- Logon/logoff activity
- File access events
- Email activity
- Device (USB) usage

**Run each cell one at a time using Shift + Enter**

**Output files will be saved to:**
- `data/logon.csv`
- `data/file.csv`
- `data/email.csv`
- `data/device.csv`

In [ ]:
# CELL 1: Imports and Configuration
# Run this first — sets everything up

import pandas as pd
import numpy as np
import os
from datetime import datetime, timedelta
import random

# Reproducibility: same seed = same data every run
np.random.seed(42)
random.seed(42)

# Configuration
NUM_USERS       = 500
NUM_DAYS        = 90
MALICIOUS_USERS = 15
START_DATE      = datetime(2023, 1, 1)
OUTPUT_DIR      = "data"

# Create output folder if it doesn't exist
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Generate user IDs: U0001, U0002, ..., U0500
all_users = [f"U{str(i).zfill(4)}" for i in range(1, NUM_USERS + 1)]

# Randomly pick users who will behave suspiciously
bad_users = random.sample(all_users, MALICIOUS_USERS)

print(f"Total users        : {NUM_USERS}")
print(f"Malicious users    : {len(bad_users)}")
print(f"Days of activity   : {NUM_DAYS}")
print(f"Output folder      : {OUTPUT_DIR}/")
print("\nSetup complete! Run the next cell.")

In [ ]:
# CELL 2: Helper Function
# This generates random timestamps — normal hours or suspicious after-hours

def rand_time(date, after_hours=False):
    """
    Returns a random datetime on the given date.
    after_hours=True = evenings and weekends (suspicious pattern).
    after_hours=False = normal business hours (7am to 6pm).
    """
    if after_hours:
        hour = random.choice(list(range(19, 24)) + list(range(0, 6)))
    else:
        hour = random.randint(7, 18)
    minute = random.randint(0, 59)
    second = random.randint(0, 59)
    return date + timedelta(hours=hour, minutes=minute, seconds=second)

print("Helper function ready! Run the next cell.")

In [ ]:
# CELL 3: Generate Logon Data
# This may take 30-60 seconds — normal for 500 users x 90 days

print("Generating logon data... please wait...")
logon_rows = []
pcs = [f"PC{str(i).zfill(4)}" for i in range(1, 201)]

for day_offset in range(NUM_DAYS):
    current_date = START_DATE + timedelta(days=day_offset)
    is_weekend = current_date.weekday() >= 5

    for user in all_users:
        is_bad = user in bad_users

        if is_weekend:
            login_prob = 0.60 if is_bad else 0.10
        else:
            login_prob = 0.95

        if random.random() > login_prob:
            continue

        num_logins = random.randint(2, 6) if is_bad and is_weekend else random.randint(1, 3)

        for _ in range(num_logins):
            ts = rand_time(current_date, after_hours=(is_bad and random.random() < 0.4))
            pc = random.choice(pcs)
            if is_bad and random.random() < 0.3:
                pc = f"PC{str(random.randint(180, 200)).zfill(4)}"

            logon_rows.append({
                "id":       f"L{len(logon_rows):07d}",
                "date":     ts.strftime("%m/%d/%Y %H:%M:%S"),
                "user":     user,
                "pc":       pc,
                "activity": "Logon"
            })

logon_df = pd.DataFrame(logon_rows)
logon_df.to_csv(f"{OUTPUT_DIR}/logon.csv", index=False)
print(f"Done! {len(logon_df):,} logon events saved to data/logon.csv")
logon_df.head()

In [ ]:
# CELL 4: Generate File Access Data

print("Generating file access data... please wait...")
file_rows = []
file_types      = ["docx", "xlsx", "pdf", "txt", "pptx", "zip", "csv"]
sensitive_types = ["zip", "csv", "pdf"]

for day_offset in range(NUM_DAYS):
    current_date = START_DATE + timedelta(days=day_offset)

    for user in all_users:
        is_bad = user in bad_users
        threat_ramp = min(1.0, day_offset / NUM_DAYS)

        if is_bad:
            num_files = int(random.randint(3, 8) + threat_ramp * random.randint(10, 35))
        else:
            num_files = random.randint(0, 8)

        for _ in range(num_files):
            ts    = rand_time(current_date, after_hours=(is_bad and random.random() < 0.35))
            ftype = random.choice(sensitive_types if is_bad and random.random() < 0.5 else file_types)
            file_rows.append({
                "id":       f"F{len(file_rows):07d}",
                "date":     ts.strftime("%m/%d/%Y %H:%M:%S"),
                "user":     user,
                "filename": f"file_{random.randint(1000,9999)}.{ftype}",
                "activity": random.choice(["open", "copy", "write", "delete"])
            })

file_df = pd.DataFrame(file_rows)
file_df.to_csv(f"{OUTPUT_DIR}/file.csv", index=False)
print(f"Done! {len(file_df):,} file events saved to data/file.csv")
file_df.head()

In [ ]:
# CELL 5: Generate Email Data

print("Generating email data... please wait...")
email_rows = []
external_domains = ["gmail.com", "yahoo.com", "hotmail.com", "protonmail.com"]
internal_domain  = "company.com"

for day_offset in range(NUM_DAYS):
    current_date = START_DATE + timedelta(days=day_offset)

    for user in all_users:
        is_bad = user in bad_users
        num_emails = random.randint(5, 20) if is_bad else random.randint(2, 10)

        for _ in range(num_emails):
            ts = rand_time(current_date, after_hours=(is_bad and random.random() < 0.3))

            if is_bad and random.random() < 0.45:
                to_addr = f"external_{random.randint(1,50)}@{random.choice(external_domains)}"
            else:
                to_addr = f"user_{random.randint(1,500)}@{internal_domain}"

            has_attachment = random.random() < (0.6 if is_bad else 0.2)

            email_rows.append({
                "id":          f"E{len(email_rows):07d}",
                "date":        ts.strftime("%m/%d/%Y %H:%M:%S"),
                "user":        user,
                "to":          to_addr,
                "cc":          "",
                "bcc":         "",
                "size":        random.randint(1000, 50000) if not has_attachment else random.randint(50000, 5000000),
                "attachments": random.randint(1, 5) if has_attachment else 0,
                "activity":    "Send"
            })

email_df = pd.DataFrame(email_rows)
email_df.to_csv(f"{OUTPUT_DIR}/email.csv", index=False)
print(f"Done! {len(email_df):,} email events saved to data/email.csv")
email_df.head()

In [ ]:
# CELL 6: Generate Device/USB Data

print("Generating device/USB data... please wait...")
device_rows = []

for day_offset in range(NUM_DAYS):
    current_date = START_DATE + timedelta(days=day_offset)

    for user in all_users:
        is_bad = user in bad_users
        use_usb = random.random() < (0.35 if is_bad else 0.03)
        if not use_usb:
            continue

        ts = rand_time(current_date, after_hours=(is_bad and random.random() < 0.5))
        device_rows.append({
            "id":       f"D{len(device_rows):07d}",
            "date":     ts.strftime("%m/%d/%Y %H:%M:%S"),
            "user":     user,
            "pc":       f"PC{str(random.randint(1,200)).zfill(4)}",
            "activity": "Connect"
        })

device_df = pd.DataFrame(device_rows)
device_df.to_csv(f"{OUTPUT_DIR}/device.csv", index=False)
print(f"Done! {len(device_df):,} device events saved to data/device.csv")
device_df.head()

In [ ]:
# CELL 7: Summary
# Run this last to confirm everything worked

total_events = len(logon_df) + len(file_df) + len(email_df) + len(device_df)

print("=" * 50)
print("  DATA GENERATION COMPLETE")
print("=" * 50)
print(f"  Total events generated : {total_events:,}")
print(f"  Logon events           : {len(logon_df):,}")
print(f"  File events            : {len(file_df):,}")
print(f"  Email events           : {len(email_df):,}")
print(f"  Device events          : {len(device_df):,}")
print(f"  Simulated insider users: {len(bad_users)}")
print("=" * 50)
print("\nAll CSV files saved to the data/ folder.")
print("Next step: Open notebook 02_preprocess.ipynb")